# Modified GBM — model specification

**This notebook is the documentation source of truth.** The PDF `V4_modified_gbm_model.pdf` in the parent folder is the same text.

Modified GBM is a discrete-time replacement for geometric Brownian motion used in the V4 1.5-year fixed 10,000-path study.

## 1. Idea

Standard GBM draws each log-return from one normal $N(\mu\Delta t,\sigma^{2}\Delta t)$. Signs are independent and up/down sizes share one volatility.

Modified GBM keeps the price map $S_{t+1}=S_t e^{r_t}$ but splits $r_t$ using the eight parameters in §4: $P(U\mid U)$, $P(D\mid D)$, $P(U\mid D)$, $P(D\mid U)$, $\mu_U$, $\sigma_U$, $\mu_D$, $\sigma_D$.

1. **Direction.** Given the last non-zero sign, draw the next sign from:
   - $P(U\mid U)$ — probability the next bar is up if the last bar was up. Use after an up bar.
   - $P(D\mid D)$ — probability the next bar is down if the last bar was down. Use after a down bar.
   - $P(U\mid D)$ — probability the next bar is up if the last bar was down. Use after a down bar.
   - $P(D\mid U)$ — probability the next bar is down if the last bar was up. Use after an up bar.
2. **Magnitude.** On an up bar the size is $|N(\mu_U,\sigma_U^{2})|$; on a down bar it is $|N(\mu_D,\sigma_D^{2})|$. So $\mu_U,\sigma_U$ are the typical up-move size and spread, and $\mu_D,\sigma_D$ are the down-move analogues.
3. **Price.** $r_t=+m_t$ on $U$ and $r_t=-m_t$ on $D$, then $S\leftarrow S e^{r_t}$.

It is return-based. Option quotes are not used in calibration. American calls are priced by LSM on risk-neutral paths.

It is **not** Heston (no variance diffusion), **not** Merton (no jumps), **not** GARCH (no $\omega,\alpha,\beta$), and **not** hidden-state regime-switching (the state is the previous observed sign).

## 2. Estimation

On each lookback window of log-returns $R_s=\ln(S_s/S_{s-1})$:

- Drop zeros. Count consecutive sign pairs. Laplace-smoothed
  $\hat{P}(U\mid U)=(n_{UU}+1/2)/(n_{\mathrm{from\ }U}+1)$, and the analogue for $\hat{P}(D\mid D)$.
- $\mu_U,\sigma_U$ (resp. $\mu_D,\sigma_D$) = mean and sample SD of $|R|$ on up (resp. down) bars.
- `last_up` = sign of the last non-zero lookback return (starts the simulator).

Fixed: one 18-month window ending at the first session of each 1-year evaluation window. Parameters dated $t_0$ are held for every Monday contract in that window and are used only after $t_0$.

## 3. Simulation

**P-measure** (stock PDF): no drift shift. Path cloud $n=10000$, seed 42. Reported path = p50 vs realized adj-close.

**Q-measure** (decision / moneyness PDFs): after drawing $r$, shift
$r\leftarrow r+(r_f\Delta t-\log\mathbb{E}e^{r})$ so $\mathbb{E}e^{r}=e^{r_f\Delta t}$, $\Delta t=1/252$. Then Longstaff–Schwartz on the same Monday ATM listed-call sample as every other model.

## 4. Parameters in the estimation PDF

The quantities used in §1, written once per regime (the single fixed lookback):

$P(U\mid U),\ P(D\mid D),\ P(U\mid D),\ P(D\mid U),\ \mu_U,\ \sigma_U,\ \mu_D,\ \sigma_D$.

## 5. Where results are

- Seven-model ranking: `V4_1p5y_fixed_empirical_study.pdf`
- Return-based group (includes Modified GBM): `V4_1p5y_fixed_empirical_study_return_based.pdf`
- Parameters / stock / moneyness: the matching `V4_1p5y_fixed_*.pdf` files in the parent folder
- Code: `V4-Models_result/modified gbm notebook/20*_modified_gbm.ipynb`
